# J-lens / Antidoom baseline — `LiquidAI/LFM2-2.6B`

**Not** the blog's private early LFM2.5-2.6B. **Not** Antidoom-trained.

| Setting | Value |
|---|---|
| Model | `LiquidAI/LFM2-2.6B` |
| Thinking | leave **default** (do not force-disable) |
| Prompts | 200 stratified antidoom-mix, seed=42 |
| max_new_tokens | **4000** |
| Backend | **vLLM** (`fp8` → fallback `bfloat16`) |

**Important:** Re-run cells from the top after Runtime → Restart. Cell 1 hard-resets `/content/j-lens` to GitHub `main` so you never stay on a stale commit.

In [ ]:
# Cell 1 — HARD RESET repo to latest GitHub main (fixes "nothing changes")
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Mithilyaganti/jlens-doom-loop-analysis.git"
PROJECT = Path("/content/j-lens")
REQUIRED_REV_PREFIX = "414b018"  # must be this or newer

def run(cmd, **kw):
    print("+", " ".join(cmd))
    return subprocess.run(cmd, check=True, text=True, capture_output=True, **kw)

# Wipe and re-clone so vendor/ dirt never blocks git pull
if PROJECT.exists():
    print("Removing stale", PROJECT)
    shutil.rmtree(PROJECT, ignore_errors=True)

print("Cloning fresh", REPO_URL)
run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)])

os.chdir(PROJECT)
rev = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True, cwd=PROJECT,
).stdout.strip()
print("PROJECT", PROJECT)
print("git", rev)

loading = (PROJECT / "jspace" / "loading.py").read_text(encoding="utf-8")
assert "load_stack_tokenizer_only" in loading, (
    f"Clone is too old (git={rev}). Missing tokenizer-only vLLM fix."
)
print("OK: load_stack_tokenizer_only present")
print("prompt sample", (PROJECT / "results" / "prompt_sample_ids.json").is_file())

In [ ]:
# Cell 2 — deps + jlens outside the git repo (so it never dirties /content/j-lens)
import os, sys, subprocess, shutil
from pathlib import Path

PROJECT = Path("/content/j-lens")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

%pip install -q -U "pandas>=2.1,<2.4" transformers accelerate bitsandbytes datasets huggingface_hub \
    scipy statsmodels tqdm pyyaml safetensors sentencepiece matplotlib seaborn

try:
    %pip install -q vllm
    print("vllm ok")
except Exception as e:
    print("vllm install failed:", e)

# Keep vendor OUTSIDE the git tree so hard-reset/re-clone stays clean
JLENS_REPO = "https://github.com/eliebak/open-jlens-data.git"
JLENS_ROOT = Path("/content/open-jlens-data")
JLENS_PKG = JLENS_ROOT / "code" / "jacobian-lens"
MARKER = JLENS_PKG / "jlens" / "__init__.py"

if not MARKER.is_file():
    if JLENS_ROOT.exists():
        shutil.rmtree(JLENS_ROOT, ignore_errors=True)
    print("Cloning open-jlens-data →", JLENS_ROOT)
    subprocess.run(["git", "clone", "--depth", "1", JLENS_REPO, str(JLENS_ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(JLENS_PKG)], check=True)

if str(JLENS_PKG) not in sys.path:
    sys.path.insert(0, str(JLENS_PKG))

import jlens
print("jlens OK", jlens.__file__)

import torch
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

# Prove the vLLM tokenizer-only path exists in THIS clone
assert "load_stack_tokenizer_only" in (PROJECT / "jspace" / "loading.py").read_text()
print("OK: ready for baseline")

In [ ]:
# Cell 3 — env + smoke-test stack (must print Tokenizer-only, NOT Loading model)
import os, sys
from pathlib import Path

PROJECT = Path("/content/j-lens")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, "/content/open-jlens-data/code/jacobian-lens")

os.environ["JLENS_MODEL"] = "LiquidAI/LFM2-2.6B"
os.environ["JLENS_BASELINE_PROMPTS"] = "200"
os.environ["JLENS_SAMPLE_SEED"] = "42"
os.environ["JLENS_MAX_NEW_TOKENS"] = "4000"
os.environ["JLENS_TEMPERATURE"] = "0.01"
os.environ["JLENS_BACKEND"] = "vllm"
os.environ["JLENS_VLLM_DTYPE"] = "fp8"
os.environ["JLENS_MAX_MODEL_LEN"] = "6000"
os.environ["JLENS_HF_QUANTIZE"] = "0"
os.environ["PYTHONPATH"] = str(PROJECT)

from jspace.loading import load_stack
stack = load_stack(require_lens=False)
print("generation_backend", stack.generation_backend)
print("model is None (expected for vLLM)", stack.model is None)
print("tokenizer", type(stack.tokenizer).__name__)
assert stack.model is None, "Expected tokenizer-only stack; still loading HF model — wrong code"
print("SMOKE TEST PASSED")

In [ ]:
# Cell 4 — full 200-prompt baseline (hours on T4)
import os
from pathlib import Path

PROJECT = Path("/content/j-lens")
os.chdir(PROJECT)
os.environ["JLENS_MODEL"] = "LiquidAI/LFM2-2.6B"
os.environ["JLENS_BASELINE_PROMPTS"] = "200"
os.environ["JLENS_SAMPLE_SEED"] = "42"
os.environ["JLENS_MAX_NEW_TOKENS"] = "4000"
os.environ["JLENS_TEMPERATURE"] = "0.01"
os.environ["JLENS_BACKEND"] = "vllm"
os.environ["JLENS_VLLM_DTYPE"] = "fp8"
os.environ["JLENS_MAX_MODEL_LEN"] = "6000"
os.environ["JLENS_HF_QUANTIZE"] = "0"
os.environ["PYTHONPATH"] = str(PROJECT)

!python /content/j-lens/scripts/run_lfm2_26b.py

In [ ]:
# Cell 5 — report + progress
import json
from pathlib import Path

os.chdir("/content/j-lens")
summary = Path("results/baseline_pass_summary.json")
report = Path("results/RUN_REPORT_lfm2-2.6b_antidoom_mix_200.md")
ckpt = Path("results/checkpoints/baseline_pass_lfm2-2.6b.json")

if ckpt.is_file():
    n = len(json.loads(ckpt.read_text())["completed_prompt_ids"])
    print(f"checkpoint progress: {n}/200")
if summary.is_file():
    s = json.loads(summary.read_text())
    print(f"loops: {s.get('n_loop')} rate={s.get('loop_rate', 0):.1%} backend={s.get('backend')}")
if report.is_file():
    print("\n--- REPORT ---\n")
    print(report.read_text())
else:
    !python scripts/write_lfm_report.py
    if report.is_file():
        print(report.read_text())

## After baseline

Exp1–3 need a fitted lens at `lenses/lfm2-2.6b.pt` (not available yet). Baseline generation + loop detection does **not** need it.

Download results from Colab when done — they are **not** auto-synced to your laptop.